# Preprocess Finance and Banking datasets

This notebook converts the downloaded Hugging Face datasets into query-passage pairs and inserts them into one PostgreSQL table: `finance`.

Sources used for training:
- MTEB Banking77 train split: queries with the same intent are paired together.
- FinQA train split: each question is paired with its gold evidence and table.
- FinanceBench: each question is paired with its financial evidence.

The test split of Banking77 and Financial PhraseBank are kept in `data/Finance`, but are not inserted into the training table.

In [ ]:
import hashlib
import math
import re
import sys
from pathlib import Path

import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "data").is_dir():
    project_dir = project_dir.parent
if not (project_dir / "data" / "Finance").is_dir():
    raise FileNotFoundError("data/Finance does not exist. Download the datasets first.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

data_dir = project_dir / "data" / "Finance"

def require_one(paths, description):
    matches = sorted(Path(path) for path in paths)
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {description}, found {len(matches)}: {matches}"
        )
    return matches[0]

banking77_train_path = require_one(
    (data_dir / "mteb_banking77").glob("data/train-*.parquet"),
    "Banking77 train parquet",
)
finqa_train_path = require_one(
    (data_dir / "finqa_parquet" / "default" / "train").glob("*.parquet"),
    "FinQA train parquet",
)
financebench_path = require_one(
    (data_dir / "financebench").glob("*.jsonl"),
    "FinanceBench JSONL",
)

print(f"Project: {project_dir}")
print(f"Banking77 train: {banking77_train_path}")
print(f"FinQA train: {finqa_train_path}")
print(f"FinanceBench: {financebench_path}")

In [ ]:
def clean_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def as_list(value) -> list:
    if value is None:
        return []
    if hasattr(value, "tolist"):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return [value]


def format_table(table) -> str:
    lines = []
    for row in as_list(table):
        cells = [clean_text(cell) for cell in as_list(row)]
        if any(cells):
            lines.append(" | ".join(cells))
    return "\n".join(lines)


def format_evidence(evidence) -> str:
    texts = []
    for item in as_list(evidence):
        if isinstance(item, dict):
            text = item.get("evidence_text", "")
        else:
            text = item
        text = clean_text(text)
        if text:
            texts.append(text)
    return "\n\n".join(texts)


def make_finqa_positive(row) -> str:
    evidence = format_evidence(row["gold_evidence"])
    table = format_table(row["table"])
    if not evidence:
        fallback_context = as_list(row["pre_text"]) + as_list(row["post_text"])
        evidence = "\n".join(clean_text(item) for item in fallback_context if clean_text(item))

    parts = []
    if evidence:
        parts.append(f"Evidence:\n{evidence}")
    if table:
        parts.append(f"Table:\n{table}")
    return "\n\n".join(parts)


def build_banking77_records(frame: pd.DataFrame) -> list[dict]:
    required_columns = {"text", "label", "label_text"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Banking77 is missing columns: {sorted(missing_columns)}")

    examples_by_label = {}
    for row in frame.itertuples(index=False):
        text = clean_text(row.text)
        label = clean_text(row.label)
        label_text = clean_text(row.label_text)
        if text and label:
            examples_by_label.setdefault(label, {"name": label_text, "texts": []})["texts"].append(text)

    records = []
    for label in sorted(examples_by_label):
        group = examples_by_label[label]
        if len(group["texts"]) < 2:
            continue
        for position, anchor in enumerate(group["texts"]):
            positive = group["texts"][(position + 1) % len(group["texts"])]
            records.append(
                {
                    "data_id": f"finance_banking77_{label}_{position:05d}",
                    "source": "MTEB Banking77",
                    "title": group["name"],
                    "topic": "intent",
                    "anchor": anchor,
                    "positive": positive,
                    "hard_negative": None,
                }
            )
    return records


def build_finqa_records(frame: pd.DataFrame) -> list[dict]:
    required_columns = {"id", "question", "gold_evidence", "table", "pre_text", "post_text"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"FinQA is missing columns: {sorted(missing_columns)}")

    records = []
    for row in frame.to_dict(orient="records"):
        anchor = clean_text(row["question"])
        positive = make_finqa_positive(row)
        if not anchor or not positive:
            continue
        source_id = clean_text(row["id"])
        record_hash = hashlib.sha1(source_id.encode("utf-8")).hexdigest()[:16]
        records.append(
            {
                "data_id": f"finance_finqa_train_{record_hash}",
                "source": "FinQA",
                "title": source_id,
                "topic": "numerical_reasoning",
                "anchor": anchor,
                "positive": positive,
                "hard_negative": None,
            }
        )
    return records


def build_financebench_records(frame: pd.DataFrame) -> list[dict]:
    required_columns = {"financebench_id", "company", "doc_name", "question_type", "question", "evidence"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"FinanceBench is missing columns: {sorted(missing_columns)}")

    records = []
    for row in frame.to_dict(orient="records"):
        anchor = clean_text(row["question"])
        positive = format_evidence(row["evidence"])
        if not anchor or not positive:
            continue
        source_id = clean_text(row["financebench_id"])
        records.append(
            {
                "data_id": f"finance_financebench_{source_id}",
                "source": "FinanceBench",
                "title": f"{clean_text(row["company"])} | {clean_text(row["doc_name"])}",
                "topic": clean_text(row["question_type"]),
                "anchor": anchor,
                "positive": positive,
                "hard_negative": None,
            }
        )
    return records

In [ ]:
banking77 = pd.read_parquet(banking77_train_path)
finqa = pd.read_parquet(finqa_train_path)
financebench = pd.read_json(financebench_path, lines=True)

banking_records = build_banking77_records(banking77)
finqa_records = build_finqa_records(finqa)
financebench_records = build_financebench_records(financebench)
records = banking_records + finqa_records + financebench_records

if not records:
    raise ValueError("No training records were prepared.")
if len({record["data_id"] for record in records}) != len(records):
    raise ValueError("Duplicate data_id values were generated.")

print(f"Banking77 pairs: {len(banking_records):,}")
print(f"FinQA pairs: {len(finqa_records):,}")
print(f"FinanceBench pairs: {len(financebench_records):,}")
print(f"Total finance records: {len(records):,}")
print("\nExample record:")
print(records[0])

In [ ]:
from sqlalchemy import func, select
from sqlalchemy.dialects.postgresql import insert as postgresql_insert

from database.models import FinanceModel
from database.sql_manager import SQL_Manager

INSERT_BATCH_SIZE = 1_000
sql_mng = SQL_Manager()
sql_mng.create_finance_model()
existing_before = sql_mng.con.scalar(
    select(func.count()).select_from(FinanceModel)
)
print(f"Existing finance records: {existing_before:,}")


def insert_batch(batch: list[dict]) -> int:
    statement = postgresql_insert(FinanceModel).values(batch)
    statement = statement.on_conflict_do_nothing(
        index_elements=["data_id"]
    ).returning(FinanceModel.data_id)
    return len(sql_mng.con.scalars(statement).all())


processed = 0
inserted = 0
try:
    for start in range(0, len(records), INSERT_BATCH_SIZE):
        batch = records[start : start + INSERT_BATCH_SIZE]
        inserted += insert_batch(batch)
        processed += len(batch)
        sql_mng.con.commit()
        print(f"Processed: {processed:,}/{len(records):,}; inserted: {inserted:,}")
except Exception:
    sql_mng.con.rollback()
    raise
finally:
    sql_mng.close()

print(f"Finished. Processed: {processed:,}; inserted: {inserted:,}; skipped: {processed - inserted:,}")

In [ ]:
sql_mng = SQL_Manager()
try:
    total = sql_mng.con.scalar(select(func.count()).select_from(FinanceModel))
    source_counts = sql_mng.con.execute(
        select(FinanceModel.source, func.count())
        .group_by(FinanceModel.source)
        .order_by(FinanceModel.source)
    ).all()
    print(f"Rows in finance table: {total:,}")
    for source, count in source_counts:
        print(f"  {source}: {count:,}")
finally:
    sql_mng.close()